<a href="https://colab.research.google.com/github/hsy0828/fcnv2-weather-demo/blob/main/run_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

GITHUB_USER = "hsy0828"
REPO_NAME = "fcnv2-weather-demo"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
TARGET_DIR = f"/content/{REPO_NAME}"

if not os.path.exists(TARGET_DIR):
    !git clone {REPO_URL} {TARGET_DIR}
else:
    %cd {TARGET_DIR}
    !git pull

%cd {TARGET_DIR}

print("=== 1. 安裝系統級庫 ===")
!apt-get update -qq && apt-get install -y -qq libeccodes-dev

print("=== 2. 安裝核心套件 ===")
!pip install --only-binary=:all: ai-models ai-models-fourcastnetv2-gfs earthkit-meteo eccodes xarray cfgrib matplotlib cartopy onnxruntime onnx || pip install ai-models ai-models-fourcastnetv2-gfs earthkit-meteo eccodes xarray cfgrib matplotlib cartopy onnxruntime onnx

print("=== 3. 降級鎖定 earthkit-data ===")
!pip install -q "earthkit-data==0.9.4" --force-reinstall --no-deps

print("=== 環境準備完成！請繼續執行 Cell 2 ===")

In [11]:
import os
import subprocess
import sys
import inspect
from ai_models.inputs import available_inputs
from ai_models.outputs import available_outputs
import ai_models_fourcastnetv2_gfs
import ai_models_fourcastnetv2_gfs.model as fcnv2_module

%cd /content/fcnv2-weather-demo

# 1. 解析 EntryPoint 載入 Input / Output 類別
inputs_dict = available_inputs()
outputs_dict = available_outputs()

# 處理 Python 新版 entrypoint 物件 (.load())
input_cls = inputs_dict["ecmwf-open-data"]
if hasattr(input_cls, "load"):
    input_cls = input_cls.load()

output_cls = outputs_dict["file"]
if hasattr(output_cls, "load"):
    output_cls = output_cls.load()

# 2. 自動取得 FourCastNetv2 模型類別
target_model_cls = getattr(fcnv2_module, "FourCastNetv2", None)
if not target_model_cls:
    for name, obj in inspect.getmembers(fcnv2_module, inspect.isclass):
        if "FourCastNet" in name or "fourcastnet" in name.lower():
            target_model_cls = obj
            break

print(f"成功載入模型類別：{target_model_cls.__name__}")

# 3. 設定預測時間步長 (6 ~ 72 小時)
lead_times = list(range(6, 78, 6))

print("=== 1. 開始執行 FourCastNetV2 氣象預測 ===")

try:
    input_handle = input_cls(
        date=20240101, time=0, lead_time=72
    )
    output_handle = output_cls(
        path="fourcastnetv2-small.grib",
    )

    # 實例化模型
    model = target_model_cls(
        input=input_handle,
        output=output_handle,
        date=20240101,
        time=0,
        lead_time=lead_times,
    )

    print("正在下載開放氣象資料與模型權重並進行推論...")
    model.run()
    print("🎉 預測成功！結果已儲存至 fourcastnetv2-small.grib")

    # 4. 自動繪圖
    print("\n=== 2. 開始執行自動繪圖 (plot_result.py) ===")
    if os.path.exists("plot_result.py"):
        subprocess.run(["python", "plot_result.py"], check=True)
        print("繪圖完成！")
    else:
        print("提示：專案目錄下未找到 plot_result.py，跳過繪圖步驟。")

except Exception as e:
    print(f"執行過程發生錯誤：{e}")
    import traceback
    traceback.print_exc()

/content/fcnv2-weather-demo
成功找到模型類別：FourCastNetv2
=== 1. 開始執行 FourCastNetV2 氣象預測 ===
執行過程發生錯誤：'EntryPoint' object is not callable


Traceback (most recent call last):
  File "/tmp/ipykernel_24329/3522761140.py", line 37, in <cell line: 0>
    input_handle = available_inputs()["ecmwf-open-data"](
        date=20240101, time=0, lead_time=72
    )
TypeError: 'EntryPoint' object is not callable
